Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

# Sequential agents

- Several nodes, each doing one job, running in a fixed order
- No branching and no loops ; `START -> A -> B -> C -> END`
- One state object travels down the line and is filled in as it goes

A small content pipeline : outline, then draft, then polish.

## 0. Install & imports

In [ ]:
# (setup cell already installs what this notebook needs)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

# LLM on the local Ollama bridge (a little temperature for varied writing)
llm = make_llm(temperature=0.3)

## 1. Define the shared state

Every node reads from and writes to this one `TypedDict`. 

#### Optional : write it first

This one is worth the time if we have it. The finished cell is directly below.

```python
class PipelineState(TypedDict, total=False):
    topic: str      # input:   the subject to write about
    outline: str    # stage 1: produced by the outliner
    # TODO: add the two remaining fields the pipeline fills in:
    #   'draft' (str) - produced by the writer (stage 2)
    #   'final' (str) - produced by the editor (stage 3)
    ____
```

In [ ]:
class PipelineState(TypedDict, total=False):
    topic: str      # input:   the subject to write about
    outline: str    # stage 1: produced by the outliner
    draft: str      # stage 2: produced by the writer
    final: str      # stage 3: produced by the editor

## 2. Define the three nodes

Each node has the signature `State -> State`: read a field, call the LLM, write a new field, return the state.

#### Optional : write it first

This one is worth the time if we have it. The finished cell is directly below.

```python
def make_outline(state: PipelineState) -> PipelineState:
    """Stage 1 - GIVEN as a worked example. Study this pattern:
    read from state, build a prompt, call the LLM, write back to state."""
    topic = state["topic"]
    prompt = f"Create a short 3-point outline for a text about: {topic}. Only list the points."
    state["outline"] = llm.invoke(prompt).content
    return state


def write_draft(state: PipelineState) -> PipelineState:
    """Stage 2 - TODO. Follow the make_outline pattern:
    read state['outline'], ask the LLM to write a short paragraph from it,
    and store the result in state['draft']."""
    outline = state["outline"]
    prompt = ____              # TODO: build the prompt using `outline`
    state["draft"] = ____      # TODO: call the LLM and store the text
    return state


def polish(state: PipelineState) -> PipelineState:
    """Stage 3 - TODO. Read state['draft'], ask the LLM to compress it to a
    single sentence, and store the result in state['final']."""
    ____                       # TODO: write the whole body (3 lines like above)
    return state
```

In [ ]:
def make_outline(state: PipelineState) -> PipelineState:
    """Stage 1 - turn the topic into a short outline."""
    topic = state["topic"]
    prompt = f"Create a short 3-point outline for a text about: {topic}. Only list the points."
    state["outline"] = llm.invoke(prompt).content
    return state


def write_draft(state: PipelineState) -> PipelineState:
    """Stage 2 - turn the outline into a paragraph."""
    outline = state["outline"]
    prompt = f"Write a short paragraph (4-5 sentences) based on this outline:\n{outline}"
    state["draft"] = llm.invoke(prompt).content
    return state


def polish(state: PipelineState) -> PipelineState:
    """Stage 3 - compress the draft into a single sentence."""
    draft = state["draft"]
    prompt = f"Summarize the following text into one clear sentence:\n{draft}"
    state["final"] = llm.invoke(prompt).content
    return state

## 3. Build the graph

Register the nodes and wire them in a straight line. This is the sequential part : fixed edges, no conditionals.

#### Optional : write it first

This one is worth the time if we have it. The finished cell is directly below.

```python
builder = StateGraph(PipelineState)

# The first node is registered for you:
builder.add_node("outline", make_outline)
# TODO: register the other two nodes -> name "draft" for write_draft, "polish" for polish
____
____

# The first edge is wired for you:
builder.add_edge(START, "outline")
# TODO: wire the rest of the sequence -> outline -> draft -> polish -> END
____
____
____

graph = builder.compile()
```

In [ ]:
builder = StateGraph(PipelineState)

# Register the three nodes
builder.add_node("outline", make_outline)
builder.add_node("draft", write_draft)
builder.add_node("polish", polish)

# Wire them in a fixed sequence: START -> outline -> draft -> polish -> END
builder.add_edge(START, "outline")
builder.add_edge("outline", "draft")
builder.add_edge("draft", "polish")
builder.add_edge("polish", END)

graph = builder.compile()

## 4. Run the pipeline

Invoke with a topic. The state flows through all three stages and comes back filled in.

#### Optional : write it first

This one is worth the time if we have it. The finished cell is directly below.

```python
# TODO: pick any topic you like
result = graph.invoke({"topic": "____"})

print("=== OUTLINE ===\n", result["outline"], "\n")
print("=== DRAFT ===\n", result["draft"], "\n")
print("=== FINAL ===\n", result["final"])
```

In [ ]:
result = graph.invoke({"topic": "The benefits of local LLMs for privacy"})

print("=== OUTLINE ===\n", result["outline"], "\n")
print("=== DRAFT ===\n", result["draft"], "\n")
print("=== FINAL ===\n", result["final"])

## 5. Bonus - watch the stages fire

`graph.stream(...)` yields after each node, so we can see the sequence execute in order rather than only the final state.

In [ ]:
# Bonus: watch each stage fire in order instead of only seeing the final state
for step in graph.stream({"topic": "Why unit tests matter"}):
    node_name = list(step.keys())[0]
    print(f"--- finished node: {node_name} ---")

### Extension ideas

- Add a 4th stage `translate` that renders `final` into Dutch, and extend the state + edges to `... -> polish -> translate -> END`.
- Give each node a different model (e.g. a bigger model for the writer) to see the sequential pattern mix models per stage.
- Replace the hand-written nodes with `create_agent` agents to see the same sequence built from full agents instead of plain functions.